# 🌋 Tu primer mapa web con Python y Folium

En este cuaderno construiremos **paso a paso** un mapa web interactivo con tres capas:

1. 🗺️ **Mapa base** (OpenStreetMap)
2. 🌋 **Capa de puntos**: volcanes de EE. UU., coloreados según su altura
3. 🌍 **Capa de polígonos**: países coloreados según su población

Además tendrá ventanas emergentes (*popups*) y un panel para activar/desactivar capas.

**Archivos que necesitas en la misma carpeta que este cuaderno:** `Volcanoes_USA.txt` y `world.json`.

## 0. Preparación

In [ ]:
# Solo la primera vez (quita el # y ejecuta):
# !pip install folium pandas

In [ ]:
import json            # para leer el archivo GeoJSON
import folium          # para crear mapas web
import pandas as pd    # para leer los datos de los volcanes

print("Versión de folium:", folium.__version__)

## 1. El mapa base

Un mapa en Folium es un objeto `folium.Map`. Parámetros clave:

- `location=[latitud, longitud]` → centro del mapa
- `zoom_start` → nivel de zoom (1 = mundo entero, 18 = calle)
- `tiles` → el estilo del mapa de fondo

Si la última línea de una celda es el mapa, Jupyter lo dibuja.

In [ ]:
mapa = folium.Map(location=[39.5, -110], zoom_start=5, tiles="OpenStreetMap")
mapa

🧪 **Prueba tú:** cambia `tiles` por `"CartoDB Voyager"`, `"CartoDB positron"` o `"CartoDB dark_matter"` y vuelve a ejecutar (aunque necesitarás una API key).

## 2. Añadir un marcador

Creamos un `folium.Marker` y lo añadimos al mapa con `.add_to(mapa)`. Latitud primero, longitud después.

In [ ]:
mapa = folium.Map(location=[46.87, -121.75], zoom_start=8)

folium.Marker(
    location=[46.8698, -121.7510],        # Monte Rainier
    popup="Monte Rainier",                 # al hacer clic
    tooltip="Pulsa para ver el nombre",    # al pasar el ratón
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(mapa)

mapa

## 3. Varios marcadores con un bucle

En lugar de repetir código, guardamos las coordenadas en una lista y recorremos con `for`.

In [ ]:
lugares = [
    ("Monte Rainier", 46.8698, -121.7510),
    ("Monte Shasta", 41.4199, -122.2010),
    ("Yellowstone", 44.4299, -110.6710),
]

mapa = folium.Map(location=[43.5, -118], zoom_start=5)

for nombre, lat, lon in lugares:
    folium.Marker([lat, lon], popup=nombre).add_to(mapa)

mapa

## 4. Leer los volcanes desde un archivo

Poner coordenadas a mano no escala. Los datos están en `Volcanoes_USA.txt`, un archivo **CSV**: columnas separadas por comas.
`pandas` lo lee y lo convierte en una tabla (*DataFrame*).

In [ ]:
volcanes = pd.read_csv("Volcanoes_USA.txt")
volcanes.head()

In [ ]:
print(volcanes.shape)                     # (filas, columnas)
volcanes[["NAME", "ELEV", "LAT", "LON"]].describe()

Nos interesan cuatro columnas: `NAME`, `ELEV` (altura en metros), `LAT` y `LON`.
Convertimos cada una en lista para poder recorrerlas a la vez.

In [ ]:
nombres = list(volcanes["NAME"])
lat = list(volcanes["LAT"])
lon = list(volcanes["LON"])
elev = list(volcanes["ELEV"])

print(nombres[:3], elev[:3])

## 5. Popups con la altura

`zip()` junta varias listas y las recorre en paralelo. En cada vuelta tenemos el nombre, latitud, longitud y altura de **un** volcán.

⚠️ El popup debe ser **texto**. Si le pasas un número directamente puedes obtener errores poco claros, así que usamos un *f-string*.

In [ ]:
mapa = folium.Map(location=[39.5, -110], zoom_start=5)

for n, lt, ln, el in zip(nombres, lat, lon, elev):
    folium.Marker(
        location=[lt, ln],
        popup=f"{n}: {el:.0f} m",
    ).add_to(mapa)

mapa

## 6. Colores según la altura

Creamos una **función** que recibe una altura y devuelve un color:

- 🟢 menos de 1000 m → verde
- 🟠 de 1000 a 3000 m → naranja
- 🔴 3000 m o más → rojo

Y cambiamos `Marker` por `CircleMarker`, que se puede colorear libremente.

In [ ]:
def color_producer(elevacion):
    if elevacion < 1000:
        return "green"
    elif elevacion < 3000:      # ya sabemos que es >= 1000
        return "orange"
    else:
        return "red"

# Comprobamos que funciona antes de usarla
assert color_producer(500) == "green"
assert color_producer(1000) == "orange"
assert color_producer(3000) == "red"

In [ ]:
mapa = folium.Map(location=[39.5, -110], zoom_start=5)

for n, lt, ln, el in zip(nombres, lat, lon, elev):
    folium.CircleMarker(
        location=[lt, ln],
        radius=6,
        popup=f"{n}: {el:.0f} m",
        color="grey",               # borde
        fill=True,
        fill_color=color_producer(el),
        fill_opacity=0.7,
    ).add_to(mapa)

mapa

## 7. Datos GeoJSON: formas geométricas

**GeoJSON** es un formato para guardar geometrías (puntos, líneas, polígonos) con sus propiedades.
`world.json` contiene un polígono por país. Lo abrimos con el módulo `json`.

> El archivo empieza con un carácter invisible llamado *BOM*. Por eso usamos `encoding="utf-8-sig"`, que lo ignora.

In [ ]:
with open("world.json", encoding="utf-8-sig") as f:
    paises = json.load(f)

print("Número de países:", len(paises["features"]))
print(paises["features"][0]["properties"])

Cada país tiene `properties` (nombre, población en `POP2005`, etc.) y una `geometry` con sus coordenadas.
Usaremos `NAME` y `POP2005`.

## 8. Capa de polígonos coloreada por población

`folium.GeoJson` dibuja los polígonos. El parámetro `style_function` es una función que Folium llama **una vez por país**; recibe el país (`feature`) y devuelve su estilo.

In [ ]:
def color_poblacion(poblacion):
    if poblacion < 10_000_000:
        return "green"
    elif poblacion < 20_000_000:
        return "orange"
    else:
        return "red"

mapa = folium.Map(location=[39.5, -110], zoom_start=3)

folium.GeoJson(
    paises,
    style_function=lambda feature: {
        "fillColor": color_poblacion(feature["properties"]["POP2005"]),
        "color": "black",       # borde
        "weight": 1,
        "fillOpacity": 0.5,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME", "POP2005"],
        aliases=["País:", "Población (2005):"],
        localize=True,          # separador de miles
    ),
).add_to(mapa)

mapa

## 9. Juntarlo todo con capas y panel de control

Un `FeatureGroup` es una **capa** que agrupa elementos. Para poder activar/desactivar las capas:

1. Creamos un `FeatureGroup` por capa y le añadimos sus elementos.
2. Añadimos los grupos al mapa.
3. **Al final** añadimos `folium.LayerControl()`. Si lo pones antes, no encuentra capas y no muestra nada.

El orden importa: añadimos primero los polígonos (abajo) y después los volcanes (encima).

In [ ]:
mapa = folium.Map(location=[39.5, -110], zoom_start=4, tiles="OpenStreetMap")

# --- Capa de población (polígonos) ---
capa_poblacion = folium.FeatureGroup(name="Población")
folium.GeoJson(
    paises,
    style_function=lambda feature: {
        "fillColor": color_poblacion(feature["properties"]["POP2005"]),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.4,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME", "POP2005"],
        aliases=["País:", "Población (2005):"],
        localize=True,
    ),
).add_to(capa_poblacion)
capa_poblacion.add_to(mapa)

# --- Capa de volcanes (puntos) ---
capa_volcanes = folium.FeatureGroup(name="Volcanes")
for n, lt, ln, el in zip(nombres, lat, lon, elev):
    folium.CircleMarker(
        location=[lt, ln],
        radius=6,
        popup=f"{n}: {el:.0f} m",
        color="grey",
        fill=True,
        fill_color=color_producer(el),
        fill_opacity=0.7,
    ).add_to(capa_volcanes)
capa_volcanes.add_to(mapa)

# --- Panel de control: SIEMPRE al final ---
folium.LayerControl(collapsed=False).add_to(mapa)

mapa

## 10. Guardar el mapa como página web

`.save()` genera un archivo HTML que se abre en cualquier navegador y se puede publicar en internet.

In [ ]:
mapa.save("mapa_volcanes.html")
print("Mapa guardado en mapa_volcanes.html")

## 🎯 Retos para practicar

1. Añade un **cuarto color** (por ejemplo, `"darkred"` para volcanes de más de 4000 m).
2. Muestra en el popup también el **tipo** de volcán (columna `TYPE`).
3. Cambia los umbrales de población de `color_poblacion` y observa qué países cambian de color.
4. Haz el radio del círculo **proporcional a la altura** (pista: `radius = el / 400`).
5. Añade una tercera capa con tus ciudades o lugares favoritos.

## 📚 Resumen

| Concepto | Herramienta |
|---|---|
| Mapa base | `folium.Map` |
| Puntos | `Marker`, `CircleMarker` |
| Datos tabulares | `pandas.read_csv` |
| Datos geométricos | `json.load` + `folium.GeoJson` |
| Estilos dinámicos | funciones + `style_function` |
| Capas | `FeatureGroup` + `LayerControl` |
| Exportar | `mapa.save("archivo.html")` |